# 行列分解によるクラス料率算定 — 分析ウォークスルー

MF / CMF を GLM・GLMM と同一のホールドアウトセル上で正直に比較する分析を、
**セル単位で追いながら**再現するノートブックです。各ステップで中間オブジェクト
（行列・分割・予測・指標）を `print` / 変数参照で確認できます。

- 既存の共有関数（`src/ratemaking.py`, `src/brazil_data_analysis_R.py`）を再利用します。
- **手動パート**（このノートの大半）は高速に追えるよう **縮小したCVグリッド** を使います。
- 最後の **フル再現パート** は本文と同じフルグリッドで `run_comparison` を回します（数分かかります）。

> 目的変数とセル露出しきい値は次のセルの `TARGET` / `CELL_EXPOSURE_MIN` で切り替えられます。

## 0. セットアップ

リポジトリのルートで起動してください（相対パス `data/...` を使うため）。

In [ ]:
import os, sys
# このノートは notebooks/ に置く想定。リポジトリのルートへ移動する。
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, 'src')   # ratemaking.py / brazil_data_analysis_R.py を import 可能に
print('cwd =', os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import ratemaking as rm
import brazil_data_analysis_R as bd
from cmfrec import CMF
import statsmodels.api as sm
import statsmodels.formula.api as smf
print('imports OK')

In [ ]:
# ---- このノートのパラメータ ----
TARGET = 'pure_premium'     # 'pure_premium'（衝突純保険料率） か 'frequency'（衝突頻度）
CELL_EXPOSURE_MIN = 100     # セル採否の露出しきい値（本文は100。50/200/300で感度も見られる）
SEED = 123
print(TARGET, CELL_EXPOSURE_MIN)

## 1. セル行列の構築

`load_cell_matrix` が CSV を読み、**車種 × 地域** の (率, 露出) 行列を作ります。
分子は衝突事故（部分衝突＋全損衝突）のみ。`pure_premium`=衝突金額/露出、`frequency`=衝突件数/露出。
露出 < しきい値 のセルは NaN（欠測）になります。

In [ ]:
rate_df, exp_df = rm.load_cell_matrix(target=TARGET, cell_exposure_min=CELL_EXPOSURE_MIN)
rate = rate_df.to_numpy(dtype=float)
exp_mat = exp_df.to_numpy(dtype=float)
models = rate_df.index.to_numpy()
areas = rate_df.columns.to_numpy()
obs = ~np.isnan(rate)
print('行列 shape :', rate.shape, '(車種 x 地域)')
print('観測セル数 :', int(obs.sum()))
vals = rate[obs]
print('率 min/median/max :', round(float(vals.min()),4), round(float(np.median(vals)),4), round(float(vals.max()),3))
rate_df.iloc[:5, :6]   # 左上をのぞく（NaN=欠測）

## 2. 露出加重とサイド情報（CMF用）

MF/CMF の損失を露出で加重（平均1に正規化）し、GLM/GLMM の `offset(log 露出)` と揃えます。
CMF は行属性（車両グループ）と列属性（人口密度）も同時に分解します。

In [ ]:
mean_exp = float(exp_mat[obs].mean())
W_full = np.nan_to_num(exp_mat, nan=0.0) / mean_exp   # per-cell weight
U_mat, I_mat, u_labels, i_labels = rm.build_side_info(rate_df)
print('W_full[obs] mean =', round(float(W_full[obs].mean()),3))
print('U (行属性: 車両グループ) :', U_mat.shape, '| I (列属性: 人口密度) :', I_mat.shape)

## 3. 学習/テスト分割（チューニングより前に確保＝リーク防止）

観測セルを 75/25 に分け、テスト側は「行(車種)も列(地域)も学習に現れるセル」だけを評価対象にします
（GLM/GLMM がそこで定義されるため）。

In [ ]:
split = rm.train_test_split(rate, ratio=0.75, seed=SEED)
X_train, X_test = split['X_train'], split['X_test']
train_mask = ~np.isnan(X_train)
rows_ok = train_mask.any(axis=1)
cols_ok = train_mask.any(axis=0)
eval_mask = (~np.isnan(X_test)) & rows_ok[:, None] & cols_ok[None, :]
er, ec = np.where(eval_mask)
act = rate[er, ec]
expw = exp_mat[er, ec]
print('学習セル :', int(train_mask.sum()), '| 評価セル :', int(eval_mask.sum()))

## 4. ハイパーパラメータ探索（縮小グリッド＝高速）

本文は `K_GRID=range(2,41)`, `LAMBDA_GRID=[0.01,0.1,1,10,20,30,50,100,1000]` を4分割CVで探索します。
ここでは追いやすさ優先で **小さなグリッド** を使います（フルは末尾のワンショットで）。
`optimize_params` は露出加重 `W=` を渡すと、最終フィットと同じ加重損失でλを較正します。

In [ ]:
K_TRY = [2, 5, 10, 20]
LAM_TRY = [1, 10, 100]
best_mf = rm.optimize_params(X_train, n_folds=4, k_values=K_TRY, lambda_values=LAM_TRY,
                             random_seed=SEED, W=W_full)
print('MF best :', best_mf)
# CMF はサイド情報の重みを固定(1.0,0.05,0.05)し、(k,λ)だけ同じ小グリッドで探索
best_cmf = rm.optimize_params(X_train, n_folds=4, k_values=K_TRY, lambda_values=LAM_TRY,
                              random_seed=SEED, W=W_full, U=U_mat, I=I_mat,
                              w_main=1.0, w_user=0.05, w_item=0.05)
print('CMF best:', best_cmf)

## 5. 4モデルを同一の評価セルで当てはめ

MF・CMF は cmfrec で（非負・非中心化・露出加重）。GLM は主効果ポアソン＋log露出オフセット。
GLMM はホールドアウトの各セルが未観測交互作用のため主効果に回帰し、予測はGLMと一致します
（本文の GLMM の限界の数値的表現）。長形式の作成には既存の `bd._long_frame` を再利用します。

In [ ]:
# --- MF ---
mf = CMF(k=best_mf['k'], lambda_=best_mf['lambda'], method='als', niter=30,
         nonneg=True, verbose=False, center=False).fit(X_train, W=W_full)
mf_pred = np.asarray(mf.predict(user=er, item=ec), dtype=float)

# --- CMF (side info) ---
cmf = CMF(k=best_cmf['k'], lambda_=best_cmf['lambda'], method='als', niter=30,
          nonneg=True, verbose=False, center=False,
          w_main=1.0, w_user=0.05, w_item=0.05).fit(X_train, W=W_full, U=U_mat, I=I_mat)
cmf_pred = np.asarray(cmf.predict(user=er, item=ec), dtype=float)

# --- GLM (main effects, Poisson, offset log exposure) ---
tr, tc = np.where(train_mask)
train_long = bd._long_frame(models, areas, tr, tc, rate, exp_mat)
test_long = bd._long_frame(models, areas, er, ec, rate, exp_mat)
le_mean = float(np.log(train_long['exposure']).mean())
train_long['log_exposure'] = np.log(train_long['exposure']) - le_mean
test_long['log_exposure'] = np.log(test_long['exposure']) - le_mean
glm = smf.glm('claim ~ C(VehModel) + C(Area)', data=train_long,
              family=sm.families.Poisson(), offset=train_long['log_exposure']).fit()
glm_pred = (glm.predict(test_long, offset=test_long['log_exposure']) / test_long['exposure']).to_numpy()

# --- GLMM: 未観測交互作用 -> 主効果に回帰 = GLM と一致 ---
glmm_pred = glm_pred
print('fitted. 予測ベクトル長 =', len(mf_pred), len(cmf_pred), len(glm_pred))

## 6. 指標テーブル（本文の表4.5.1相当）

RMSE・露出加重RMSE・（総事故スケールの）ポアソン逸脱度を横並びで。低いほど良い。

In [ ]:
def metrics(pred):
    return {
        'RMSE': float(np.sqrt(np.mean((pred - act) ** 2))),
        'wRMSE(exposure)': float(rm.weighted_rmse(pred, act, expw)),
        'PoissonDeviance': float(rm.poisson_deviance(act * expw, pred * expw)),
    }
comparison = pd.DataFrame({
    'MF (weighted)': metrics(mf_pred),
    'CMF (side info)': metrics(cmf_pred),
    'GLM': metrics(glm_pred),
    'GLMM': metrics(glmm_pred),
}).T
comparison

## 7. 露出で層別（本文の表4.5.2相当）

評価セルを露出の中央値でスパース/密に分け、各層で露出加重RMSEとポアソン逸脱度を見ます。

In [ ]:
median_exp = float(np.median(expw))
strata = {'sparse (exp<median)': expw < median_exp, 'dense (exp>=median)': expw >= median_exp}
preds = {'MF': mf_pred, 'CMF': cmf_pred, 'GLM': glm_pred, 'GLMM': glmm_pred}
rows = []
for sname, smask in strata.items():
    for mname, pred in preds.items():
        rows.append({'stratum': sname, 'model': mname, 'n': int(smask.sum()),
                     'wRMSE': rm.weighted_rmse(pred[smask], act[smask], expw[smask]),
                     'PoissonDev': rm.poisson_deviance(act[smask]*expw[smask], pred[smask]*expw[smask])})
pd.DataFrame(rows)

## 8. 予測 vs 真値の散布図（インライン表示）

対角線上に集中していれば予測が真値に整合。`ratemaking.visualize_scatter_plot` はファイル保存用
（描画後に閉じる）なので、ここではインライン用に自前で描きます。

In [ ]:
lim = float(np.nanpercentile(act, 99)) or 1.0
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for axi, (nm, pr) in zip(axes, [('MF', mf_pred), ('CMF', cmf_pred), ('GLM', glm_pred)]):
    axi.scatter(act, pr, s=10, alpha=0.5, color='black')
    axi.plot([0, lim], [0, lim], color='red')
    axi.set_xlim(0, lim); axi.set_ylim(0, lim)
    axi.set_xlabel('true'); axi.set_ylabel('pred'); axi.set_title(nm)
plt.tight_layout(); plt.show()

## 9. デバッグのヒント（IPython / Jupyter）

- ある関数の内部を止めて調べたい: 関数本体に `breakpoint()` を置く → 実行するとそのセルで pdb に入る。
- 例外が出た直後に事後解析: 別セルで `%debug` を実行すると落ちた地点の pdb が開く。
- 任意地点で対話に入る: `from IPython import embed; embed()` を仕込む。
- 変数の中身: `rate_df.describe()`, `comparison`, `np.where(eval_mask)` などをセルで評価。
- 例えば `optimize_params` の中を追うなら、`ratemaking.py` の該当行に `breakpoint()` を入れて
  このノートの Step 4 を再実行。

## 10. フル再現（本文と同じ設定・数分かかる）

`prepare_data` + `run_comparison(write=False)` はフルグリッドで探索し、本文の表と同じ結果を返します。
`write=False` なので正本の `docs/` や図は上書きしません。`ctx['comparison']` と `ctx['strat']` に結果が入ります。

In [ ]:
# 数分かかるので必要なときだけ実行（コメントを外す）。
# pp, pp_mat, e_mat, obs_cells, W, U, I = bd.prepare_data(target=TARGET, cell_exposure_min=CELL_EXPOSURE_MIN)
# best, ctx = bd.run_comparison(pp, pp_mat, e_mat, W, U, I, target=TARGET, write=False)
# print('best (k,lambda):', best)
# display(ctx['comparison'])
# display(ctx['strat'])